# Tamil Handwritten OCR: Kaggle GPU Multi-Split Training Pipeline

This notebook trains and evaluates the **uTHCD Tamil Handwritten Character Recognition CNN** sequentially across all 4 split configurations:
1. **70/30 (Stage A)**: 55,000 train / 7,870 val / 28,080 test
2. **80/20 (Stage B)**: 64,584 train / 7,176 val / 19,190 test
3. **85/15 (Stage C)**: 69,638 train / 7,738 val / 13,574 test
4. **90/10 (Stage D)**: 73,710 train / 8,190 val / 9,050 test

### Rules & Safeguards:
- **Zero Data Leakage**: Validation subsets are drawn strictly from each split's training partition. Test sets are evaluated ONLY after model selection.
- **Resumable**: Saves latest_checkpoint.pt every epoch. If interrupted by Kaggle session limits, rerunning the cell safely resumes from the exact epoch.
- **GPU Mixed-Precision (AMP)**: Accelerates convergence on Tesla P100 / T4 GPUs with PyTorch AMP.
- **Full Test Metric Suite**: Top-1, Top-3, Macro Precision, Recall, F1, and Confusion Matrix.

In [ ]:
# 1. Hardware & CUDA Environment Verification
import torch, os, sys

print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device Name:', torch.cuda.get_device_name(0))
    print('Device Count:', torch.cuda.device_count())
    props = torch.cuda.get_device_properties(0)
    print(f'VRAM Total: {props.total_memory / (1024**3):.2f} GB')
else:
    print('WARNING: GPU is not active. Enable GPU in Notebook Settings (Settings -> Accelerator -> GPU T4 or P100).')

In [ ]:
# 2. Clone/Update Project Repository
REPO_DIR = '/kaggle/working/tamil-handwritten-unicode'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/kamalakannan675/tamil-handwritten-unicode.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Active working directory:', os.getcwd())

In [ ]:
# 3. Discover and Map Dataset Files
import os, h5py

SEARCH_ROOTS = ['/kaggle/input', 'data', f'{REPO_DIR}/data']
SPLIT_FILES = {
    '70_30': 'hdf5_uTHCD_compressed.h5',
    '80_20': 'hdf5_uTHCD_80_20_split.h5',
    '85_15': 'hdf5_uTHCD_85_15_split.h5',
    '90_10': 'hdf5_uTHCD_90_10_split.h5'
}

DATASET_MAP = {}
print('Scanning for HDF5 dataset files...')
for split_name, filename in SPLIT_FILES.items():
    found_path = None
    for root in SEARCH_ROOTS:
        if os.path.exists(root):
            for dirpath, _, filenames in os.walk(root):
                if filename in filenames:
                    found_path = os.path.join(dirpath, filename)
                    break
        if found_path:
            break
    if found_path:
        DATASET_MAP[split_name] = found_path
        sz_mb = os.path.getsize(found_path) / (1024 * 1024)
        print(f'  [FOUND] {split_name}: {found_path} ({sz_mb:.2f} MB)')
    else:
        print(f'  [MISSING] {split_name}: {filename} (attach as Kaggle dataset if not in git)')

print('Configured Dataset Map:', DATASET_MAP)

In [ ]:
# 4. Quick GPU Smoke Test (Verifies CUDA forward/backward and AMP)
first_split = next(iter(DATASET_MAP.keys()), None)
if first_split:
    test_path = DATASET_MAP[first_split]
    print(f'Running GPU smoke test with {test_path}...')
    !python model/train.py --dataset-path "{test_path}" --smoke-test --checkpoint-dir outputs/gpu_smoke_test
    print('GPU Smoke test passed!')
else:
    print('No dataset file found to run smoke test.')

In [ ]:
# 5. STAGE A: Train and Evaluate 70/30 Model
split_id = '70_30'
if split_id in DATASET_MAP:
    ds_path = DATASET_MAP[split_id]
    out_dir = f'outputs/experiments/{split_id}'
    os.makedirs(out_dir, exist_ok=True)
    latest_ckpt = os.path.join(out_dir, 'latest_checkpoint.pt')
    resume_arg = f'--resume "{latest_ckpt}"' if os.path.exists(latest_ckpt) else ''
    
    print(f'=== Starting STAGE A: {split_id} ===')
    !python model/train.py \
        --dataset-path "{ds_path}" \
        --epochs 20 \
        --batch-size 64 \
        --learning-rate 0.001 \
        --augmentation light \
        --experiment-name {split_id} \
        --checkpoint-dir outputs/experiments \
        --patience 5 \
        --min-epochs 5 \
        {resume_arg}
        
    best_ckpt = os.path.join(out_dir, 'best_model.pt')
    print(f'=== Evaluating STAGE A: {split_id} on Test Partition ===')
    !python model/evaluate.py \
        --checkpoint-path "{best_ckpt}" \
        --dataset-path "{ds_path}" \
        --max-test 0 \
        --batch-size 128 \
        --output-dir "{out_dir}" \
        --device auto
else:
    print(f'Dataset for {split_id} not available.')

In [ ]:
# 6. STAGE B: Train and Evaluate 80/20 Model
split_id = '80_20'
if split_id in DATASET_MAP:
    ds_path = DATASET_MAP[split_id]
    out_dir = f'outputs/experiments/{split_id}'
    os.makedirs(out_dir, exist_ok=True)
    latest_ckpt = os.path.join(out_dir, 'latest_checkpoint.pt')
    resume_arg = f'--resume "{latest_ckpt}"' if os.path.exists(latest_ckpt) else ''
    
    print(f'=== Starting STAGE B: {split_id} ===')
    !python model/train.py \
        --dataset-path "{ds_path}" \
        --epochs 20 \
        --batch-size 64 \
        --learning-rate 0.001 \
        --augmentation light \
        --experiment-name {split_id} \
        --checkpoint-dir outputs/experiments \
        --patience 5 \
        --min-epochs 5 \
        {resume_arg}
        
    best_ckpt = os.path.join(out_dir, 'best_model.pt')
    print(f'=== Evaluating STAGE B: {split_id} on Test Partition (19,190 samples) ===')
    !python model/evaluate.py \
        --checkpoint-path "{best_ckpt}" \
        --dataset-path "{ds_path}" \
        --max-test 0 \
        --batch-size 128 \
        --output-dir "{out_dir}" \
        --device auto
else:
    print(f'Dataset for {split_id} not available.')

In [ ]:
# 7. STAGE C: Train and Evaluate 85/15 Model
split_id = '85_15'
if split_id in DATASET_MAP:
    ds_path = DATASET_MAP[split_id]
    out_dir = f'outputs/experiments/{split_id}'
    os.makedirs(out_dir, exist_ok=True)
    latest_ckpt = os.path.join(out_dir, 'latest_checkpoint.pt')
    resume_arg = f'--resume "{latest_ckpt}"' if os.path.exists(latest_ckpt) else ''
    
    print(f'=== Starting STAGE C: {split_id} ===')
    !python model/train.py \
        --dataset-path "{ds_path}" \
        --epochs 20 \
        --batch-size 64 \
        --learning-rate 0.001 \
        --augmentation light \
        --experiment-name {split_id} \
        --checkpoint-dir outputs/experiments \
        --patience 5 \
        --min-epochs 5 \
        {resume_arg}
        
    best_ckpt = os.path.join(out_dir, 'best_model.pt')
    print(f'=== Evaluating STAGE C: {split_id} on Test Partition (13,574 samples) ===')
    !python model/evaluate.py \
        --checkpoint-path "{best_ckpt}" \
        --dataset-path "{ds_path}" \
        --max-test 0 \
        --batch-size 128 \
        --output-dir "{out_dir}" \
        --device auto
else:
    print(f'Dataset for {split_id} not available.')

In [ ]:
# 8. STAGE D: Train and Evaluate 90/10 Model
split_id = '90_10'
if split_id in DATASET_MAP:
    ds_path = DATASET_MAP[split_id]
    out_dir = f'outputs/experiments/{split_id}'
    os.makedirs(out_dir, exist_ok=True)
    latest_ckpt = os.path.join(out_dir, 'latest_checkpoint.pt')
    resume_arg = f'--resume "{latest_ckpt}"' if os.path.exists(latest_ckpt) else ''
    
    print(f'=== Starting STAGE D: {split_id} ===')
    !python model/train.py \
        --dataset-path "{ds_path}" \
        --epochs 20 \
        --batch-size 64 \
        --learning-rate 0.001 \
        --augmentation light \
        --experiment-name {split_id} \
        --checkpoint-dir outputs/experiments \
        --patience 5 \
        --min-epochs 5 \
        {resume_arg}
        
    best_ckpt = os.path.join(out_dir, 'best_model.pt')
    print(f'=== Evaluating STAGE D: {split_id} on Test Partition (9,050 samples) ===')
    !python model/evaluate.py \
        --checkpoint-path "{best_ckpt}" \
        --dataset-path "{ds_path}" \
        --max-test 0 \
        --batch-size 128 \
        --output-dir "{out_dir}" \
        --device auto
else:
    print(f'Dataset for {split_id} not available.')

In [ ]:
# 9. Consolidated Multi-Split Benchmark Comparison
!python scripts/compile_multi_split_comparison.py


In [ ]:
# 10. Archive all outputs for download
!zip -r /kaggle/working/tamil_ocr_all_splits_artifacts.zip outputs/experiments outputs/reports
print('Outputs archived to /kaggle/working/tamil_ocr_all_splits_artifacts.zip')